# Alternative Credit Scoring — ML Model

A presentation-ready notebook for the project's ML contribution.

**Pipeline:** Data → Validation → Preprocessing → Logistic Regression baseline → XGBoost → Evaluation → Feature Importance → SHAP → Probability of Default → 0–1000 Credit Score → Risk Tier → CSV/model artifacts.

**Models:** Logistic Regression (baseline) and XGBoost (primary nonlinear model).

**Target:** `borrower_outcome` — 0 = Successful Borrower, 1 = Defaulted.

**Important:** IDs, `match_cost`, `risk_index`, and duplicate target fields are excluded to avoid leakage. Delinquency features are retained.


In [1]:
# 1. Imports
# If necessary: %pip install pandas numpy scikit-learn xgboost shap matplotlib joblib

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
from xgboost import XGBClassifier
import shap

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
RANDOM_STATE = 42
print("Libraries loaded successfully.")


ModuleNotFoundError: No module named 'xgboost'

## 2. Configuration

Change only `DATA_PATH` if your CSV has a different location/name.

In [ ]:
BASE_DIR = Path.cwd()

DATA_PATH = (
    BASE_DIR / "data" / "processed" /
    "ml_training_dataset_model_A_realistic_calibrated_v2_5500.csv"
)

OUTPUT_DIR = BASE_DIR / "ml_notebook_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_PATH)
print("Outputs :", OUTPUT_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Update DATA_PATH above."
    )


## 3. Load and validate data

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

print("\nMissing values:")
missing = df.isnull().sum()
display(missing[missing > 0].to_frame("missing_count"))
if missing.sum() == 0:
    print("No missing values found.")

TARGET = "borrower_outcome"
print("\nTarget distribution:")
display(df[TARGET].value_counts().to_frame("count"))
display((df[TARGET].value_counts(normalize=True)*100).round(2).to_frame("percentage"))


## 4. Feature selection

The model uses 22 legitimate borrower features covering lifestyle, spending behaviour, repayment discipline, delinquency and behavioural signals.

Excluded: `user_id`, `applicant_id`, `match_cost`, `risk_index`, `target_class`.

`risk_index` is excluded because it is an intermediary variable involved in synthetic target generation and would cause target leakage.


In [ ]:
FEATURES = [
    "age", "education_level", "employment_status", "monthly_income",
    "city_tier", "months_at_job", "housing", "rent_on_time_months",
    "digital_payment_rate", "education",
    "monthly_spend", "essential_pct", "cashflow_volatility", "savings_days",
    "on_time_rate", "dti", "credit_util",
    "delinq_90plus", "delinq_60plus", "delinq_30plus",
    "positive_habits", "risk_flags"
]

excluded = [
    "user_id", "applicant_id", "match_cost",
    "risk_index", "target_class"
]

missing_features = [f for f in FEATURES + [TARGET] if f not in df.columns]
if missing_features:
    raise ValueError(f"Missing required columns: {missing_features}")

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

CATEGORICAL_FEATURES = [
    "education_level", "employment_status", "housing", "education"
]
NUMERICAL_FEATURES = [f for f in FEATURES if f not in CATEGORICAL_FEATURES]

print("ML features:", len(FEATURES))
print("Categorical:", len(CATEGORICAL_FEATURES))
print("Numerical  :", len(NUMERICAL_FEATURES))


## 5. Preprocessing

Numerical features use median imputation + standardization. Categorical features use most-frequent imputation + one-hot encoding.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, NUMERICAL_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])


## 6. Stratified train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


## 7. Logistic Regression baseline

Chosen as a simple, interpretable binary-classification baseline that directly produces probability estimates.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)

logistic_probability = logistic_model.predict_proba(X_test)[:, 1]
logistic_prediction = (logistic_probability >= 0.50).astype(int)

print("Logistic Regression trained.")


## 8. XGBoost primary model

Chosen for structured/tabular data and its ability to model nonlinear relationships and feature interactions.

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.035,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.10,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist"
)

xgb_model.fit(X_train_encoded, y_train)

xgb_probability = xgb_model.predict_proba(X_test_encoded)[:, 1]
xgb_prediction = (xgb_probability >= 0.50).astype(int)

print("Encoded shape:", X_train_encoded.shape)
print("XGBoost trained.")


## 9. Model comparison

In [ ]:
def evaluate_model(name, y_true, predictions, probabilities):
    cm = confusion_matrix(y_true, predictions)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions),
        "recall": recall_score(y_true, predictions),
        "f1": f1_score(y_true, predictions),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "brier_score": brier_score_loss(y_true, probabilities),
        "true_negative": int(cm[0,0]),
        "false_positive": int(cm[0,1]),
        "false_negative": int(cm[1,0]),
        "true_positive": int(cm[1,1]),
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_prediction, logistic_probability),
    evaluate_model("XGBoost", y_test, xgb_prediction, xgb_probability)
])

display(results.round(4))
results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


## 10. XGBoost classification report and confusion matrix

In [ ]:
print(classification_report(
    y_test, xgb_prediction,
    target_names=["Successful Borrower", "Defaulted"]
))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_prediction,
    display_labels=["Successful", "Defaulted"],
    ax=ax
)
ax.set_title("XGBoost Confusion Matrix")
plt.tight_layout()
plt.show()


## 11. ROC and Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

RocCurveDisplay.from_predictions(
    y_test, logistic_probability,
    name="Logistic Regression", ax=axes[0]
)
RocCurveDisplay.from_predictions(
    y_test, xgb_probability,
    name="XGBoost", ax=axes[0]
)
axes[0].set_title("ROC Curve")

PrecisionRecallDisplay.from_predictions(
    y_test, logistic_probability,
    name="Logistic Regression", ax=axes[1]
)
PrecisionRecallDisplay.from_predictions(
    y_test, xgb_probability,
    name="XGBoost", ax=axes[1]
)
axes[1].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.show()


## 12. XGBoost feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(20))
feature_importance.to_csv(
    OUTPUT_DIR / "xgboost_feature_importance.csv",
    index=False
)

top = feature_importance.head(15).sort_values("importance")
plt.figure(figsize=(10,7))
plt.barh(top["feature"], top["importance"])
plt.xlabel("Importance")
plt.title("Top XGBoost Feature Importance")
plt.tight_layout()
plt.show()


## 13. SHAP explainability

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_encoded)

shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance.head(20))
shap_importance.to_csv(
    OUTPUT_DIR / "xgboost_shap_importance.csv",
    index=False
)


In [ ]:
plt.figure(figsize=(10,7))
shap.summary_plot(
    shap_values, X_test_encoded,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_summary.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(10,8))
shap.summary_plot(
    shap_values, X_test_encoded,
    feature_names=feature_names,
    show=False
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_beeswarm.png", dpi=200, bbox_inches="tight")
plt.show()


## 14. Probability of Default → Credit Score

The project scoring formula is:

**Credit Score = 1000 × (1 − Probability of Default)**

Risk tiers:
- 750–1000: Prime / Low Risk
- 650–749: Near-Prime / Medium Risk
- 550–649: Sub-Prime / Fair
- 0–549: High Risk / Poor


In [ ]:
probability_of_default = xgb_probability

credit_score = np.clip(
    1000 * (1 - probability_of_default),
    0, 1000
).round().astype(int)

def get_risk_tier(score):
    if score >= 750:
        return "Prime / Low Risk"
    elif score >= 650:
        return "Near-Prime / Medium Risk"
    elif score >= 550:
        return "Sub-Prime / Fair"
    return "High Risk / Poor"

risk_tier = [get_risk_tier(s) for s in credit_score]

score_output = pd.DataFrame({
    "probability_of_default": probability_of_default,
    "credit_score": credit_score,
    "risk_tier": risk_tier
})

display(score_output.head(10))


## 15. Score and PD distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].hist(credit_score, bins=20, edgecolor="black")
axes[0].set_title("Credit Score Distribution")
axes[0].set_xlabel("Credit Score")
axes[0].set_ylabel("Applicants")

axes[1].hist(probability_of_default, bins=20, edgecolor="black")
axes[1].set_title("Probability of Default Distribution")
axes[1].set_xlabel("Probability of Default")
axes[1].set_ylabel("Applicants")

plt.tight_layout()
plt.show()

print("Score statistics:")
display(pd.Series(credit_score, name="credit_score").describe().to_frame())

print("Risk tier distribution:")
display(pd.Series(risk_tier, name="risk_tier").value_counts().to_frame("applicants"))


## 16. Export prediction artifact

This CSV is the main standalone evidence of the ML module: applicant data → predicted PD → credit score → risk tier.

In [ ]:
prediction_output = X_test.copy()

prediction_output["actual_outcome"] = y_test.values
prediction_output["probability_of_default"] = probability_of_default
prediction_output["predicted_outcome"] = xgb_prediction
prediction_output["credit_score"] = credit_score
prediction_output["risk_tier"] = risk_tier

prediction_path = OUTPUT_DIR / "credit_score_predictions.csv"
prediction_output.to_csv(prediction_path, index=False)

display(prediction_output[
    ["actual_outcome", "probability_of_default",
     "predicted_outcome", "credit_score", "risk_tier"]
].head(15))

print("Saved:", prediction_path)


## 17. Save models and metadata

In [ ]:
xgb_bundle = {
    "preprocessor": preprocessor,
    "model": xgb_model,
    "features": FEATURES,
    "target": TARGET
}

joblib.dump(
    xgb_bundle,
    OUTPUT_DIR / "xgboost_credit_model.joblib"
)

joblib.dump(
    logistic_model,
    OUTPUT_DIR / "logistic_regression_model.joblib"
)

metadata = {
    "dataset": str(DATA_PATH),
    "records": int(len(df)),
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numerical_features": NUMERICAL_FEATURES,
    "target": TARGET,
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "primary_model": "XGBoost",
    "baseline_model": "Logistic Regression",
    "credit_score_formula": "1000 * (1 - probability_of_default)",
    "risk_tiers": {
        "750-1000": "Prime / Low Risk",
        "650-749": "Near-Prime / Medium Risk",
        "550-649": "Sub-Prime / Fair",
        "0-549": "High Risk / Poor"
    },
    "excluded_features": excluded
}

with open(OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Models and metadata saved.")


# 18. Final ML Deliverables

The notebook generates:

```text
ml_notebook_outputs/
├── credit_score_predictions.csv
├── model_comparison.csv
├── xgboost_feature_importance.csv
├── xgboost_shap_importance.csv
├── shap_summary.png
├── shap_beeswarm.png
├── xgboost_credit_model.joblib
├── logistic_regression_model.joblib
└── model_metadata.json
```

### Final pipeline

**22 alternative-credit features → preprocessing → Logistic Regression baseline + XGBoost → Probability of Default → 0–1000 Credit Score → Risk Tier → Feature/SHAP Explainability**

This ML module can remain independent of the frontend/backend integration.
